# 使用するコンペについて

In [ ]:
# コンペのURL
# zillow prize
# https://www.kaggle.com/competitions/zillow-prize-1

In [ ]:
# コンペの概要

# ■目的
# 2017年秋におけるlogerrorを予測する．
# Zestimate(モデルの予測値)と売値(実データ)に各対数を取り、
# その差分をlogerrorとする。
# logerror = log(Zestimate) - log(SalePrice)

# ■訓練データ
# 2016年の三箇所（Los Angeles，Orange and Ventura，California）の
# 全ての不動産データが提供される．訓練データは2016年10月15日以前の全ての取引と，
# それ以降のいくつかの取引を含む．

# ■テストデータ
# test data for public leaderboard
# 2016年10月15日から同年12月31日までのものである．
# test data for private leader board
# 2017年10月15日から同年12月15日まで（この期間をsales tracking periodと呼ぶ）のものである．
# 要するに以下の6個を予測する必要がある。
# October 2016 (201610), November 2016 (201611)
# December 2016 (201612), October 2017 (201710)
# November 2017 (201711), December 2017 (201712)

# ■各データファイルの概要
# properties_2016.csv：2016年の全ての不動産情報．
# properties_2017.csv：2017年の全ての不動産情報．2017年10月2日に公開予定．
# train_2016.csv：2016年1月1日から2016年12月31日までの取引情報．
# train_2017.csv：2017年1月1日から2017年9月15日までの取引情報．2017年10月2日に公開予定．
# sample_submission.csv：提出用のサンプルファイル．

# ■その他
# 当該期間において取引が行われなかった不動産は，scoreの計算に利用されない．
# 31日間のうち，複数回の取引があった場合は，
# 最初の適切な取引をscoreの計算に用いる．


#実装

dropout crossvalidation xgboost

## 必要なライブラリ

In [ ]:
# 必要なライブラリのインポート

from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import torch.optim as optim
import seaborn as sns
from tqdm import tqdm
# torch.manual_seed(1)

import torch.utils.data as data_utils
import torch.nn.init as init

import xgboost as xgb

from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler ,LabelEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

from tempfile import mkdtemp
import datetime
from dateutil.parser import parse
import inspect
from numbers import Number
import math

import zlib
import zipfile

In [ ]:
# import sys
# sys.path.append('/content/gdrive/MyDrive/my_modules')

In [ ]:
# optunaのインストール

!pip install optuna

##データセット

In [ ]:
print('Loading Properties ...')
properties2016 = pd.read_csv('/content/drive/MyDrive/kaggle/zillow_prize/datasets/properties_2016.csv', low_memory = False)
properties2017 = pd.read_csv('/content/drive/MyDrive/kaggle/zillow_prize/datasets/properties_2017.csv', low_memory = False)

print('Loading Train ...')
train2016 = pd.read_csv('/content/drive/MyDrive/kaggle/zillow_prize/datasets/train_2016_v2.csv', parse_dates=['transactiondate'], low_memory=False)
train2017 = pd.read_csv('/content/drive/MyDrive/kaggle/zillow_prize/datasets/train_2017.csv', parse_dates=['transactiondate'], low_memory=False)

In [ ]:
properties2016.info()

In [ ]:
properties2016.head(3)

In [ ]:
train2016.info()

In [ ]:
train2016.head(3)

In [ ]:
data2016 = pd.merge(train2016, properties2016, how = 'left', on = 'parcelid')
data2017 = pd.merge(train2017, properties2017, how = 'left', on = 'parcelid')

In [ ]:
data = pd.concat([data2016, data2017], axis=0, ignore_index=True)
data.head()

In [ ]:
# Column: heatingorsystemtypeid
# n_unique: 13
# 2.000000     71936
# 3.924525     62237
# 7.000000     29626
# 24.000000     1921
# 6.000000      1747
# 20.000000      201
# 13.000000      136
# 18.000000       48
# 1.000000        26
# 10.000000        5
# 11.000000        2
# 14.000000        2
# 12.000000        1

In [ ]:
data.shape

In [ ]:
data.isna().sum()

In [ ]:
# for col in ['heatingorsystemtypeid']:
#     val_cnt = data[col].value_counts()
#     print(f"Column: {col}")
#     print(f"n_unique: {data[col].nunique()}")
#     print(val_cnt)
#     print()

In [ ]:
data.columns

In [ ]:
# dupl = data.pivot_table(index = ['parcelid'], aggfunc = 'size')

In [ ]:
# data[data.parcelid == 10711910]

In [ ]:
# dupl[dupl > 1]

データセットの欠損値  
初めに、各カラムの欠損値の割合を計算します。9割の欠損値を含むカラムは削除対象ですが、最初にそれらのデータに特殊性があるかどうかを調べます。

In [ ]:
# データセットのその他の変数の欠損値の確認
na_ratio = data.isna().sum().sort_values(ascending=False)/len(data)
na_ratio

変数`missing_09` は、90% 以上の欠損値を持つカラムの格納先です。 変数`missing_09`にはオブジェクト型変数が3個、float64型変数が17個あります。

In [ ]:
missing_09 = na_ratio[na_ratio>0.9].index.tolist()
data[missing_09].dtypes

In [ ]:
target_types = list(set(data[missing_09].dtypes))
data[missing_09].dtypes.value_counts().get(target_types, 0)

カテゴリ変数`fireplaceflag`、`hashottuborspa`、`taxdelinquencyflag`に内包しているNaNはそれぞれ「不明」ではなく「ない」という意味のある値なので削除しないでNoneに置き換えてることにします。

In [ ]:
na_obj = data[missing_09].select_dtypes('object').columns
na_obj

In [ ]:
#対象のオブジェクト型変数の欠損値をNoneに置き換える
data[na_obj] = data[na_obj].fillna('None')

In [ ]:
#可視化
fig, ax = plt.subplots(ncols = 3, nrows = 1, figsize = (9,3))
for i, col in enumerate(na_obj):
    data[col].value_counts().plot.bar(ax = ax[i], color = '#d4dddd')
    ax[i].set_title(f'{col} distribution', fontsize = 10);
plt.tight_layout()
plt.show()

ここで、変数`finishedsquarefeet**`について考察してみる。これらの変数は、施工後の特定のfeetごとの物件の総面積と考えられる。以下の図のように12feet級の物件が多く、それ以外の物件は欠損値が多い、つまり広い総面積の物件の絶対数はかなり少ないことが分かる。対して変数`calculatedfinishedsquarefeet`は完成後の物件の総面積であり、これを物件の面積の指標の代表として活用するようにする。それ以外は、欠損値が多く極端であり、似たような性質を持つカラムをモデルの特徴量にすると多重共線性の可能性があるためデータセットから削除する。

In [ ]:
def plot_histogram(data, column_name, ax):
    sns.histplot(data[column_name], ax=ax)
    ax.set_title("Histogram of " + column_name)
    ax.set_xlabel("Values")
    ax.set_ylabel("Frequency")

    # 欠損値の割合を計算
    missing_percentage = data[column_name].isnull().mean() * 100

    # 欠損値の割合を表示
    ax.text(0.95, 0.95, f"Missing: {missing_percentage:.2f}%",
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(facecolor='red', alpha=0.5))

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

# column_names = [x for x in data.columns if x.startswith('finishedsquarefeet')]
to_drop = data.filter(like='squarefeet')
to_drop = to_drop.columns

for i, column_name in enumerate(to_drop):
    plot_histogram(data, column_name, axes[i])

plt.tight_layout()
plt.show()


変数`calculatedfinishedsquarefeet`以外のsquarefeet関連の変数はすべて削除する。

In [ ]:
to_drop = data.filter(like='squarefeet')
to_drop = list(to_drop.columns)
to_drop.remove('calculatedfinishedsquarefeet')
data.drop(to_drop, axis=1, inplace=True) #axis=1 の付け忘れに注意

変数`parcelid`は一意のidでありモデルの特徴量として削除しても差し支えないと考えられるので削除する。

In [ ]:
data.drop('parcelid', axis=1, inplace=True)

変数`missing_09`から先ほどの変数`to_drop`を除外する。

In [ ]:
missing_09 = list(set(missing_09) - set(to_drop))

変数`na_float`は、変数`missing_float`のうちfloat型のカラムを指す。

In [ ]:
na_float = data[missing_09].select_dtypes('float').columns
na_float
# data.drop(na_float, axis=1, inplace=True)


変数`na_float`は欠損値が9割以上を占めるfloat型の変数の集まりであり、有用性はないので削除する。

In [ ]:
data[na_float].isna().sum().sort_values(ascending=False)/len(data[na_float])

In [ ]:
data.drop(na_float, axis=1, inplace=True)

再び、過度な欠損値を含む変数を除外したデータセットの変数の欠損値の状態を見てみる。

In [ ]:
na_ratio = data.isna().sum().sort_values(ascending=False)/len(data)
na_ratio

idを含むカラムについて考察する。

In [ ]:
# "id"を含むカラムのみを抽出
id_columns = data.filter(like='id', axis=1).columns

In [ ]:
id_columns

カラム名にidとつく場合は欠損値の置き換え方に注意が必要である。なぜなら、単純に平均で置き換えると実際には使用されていないidが割り振られてしまう可能性があるからである。

idに関するカラムの一意な値の個数と欠損値の確認をすると、変数`pooltypeid7`は一種類しか値がなく、欠損値も大きいことからモデルの精度に寄与しないと考えられる。よって、削除する。

In [ ]:
# idに関するカラムの一意な値の個数と欠損値の確認
na_ratio = data[id_columns].isna().sum().sort_values(ascending=False)/len(data)
na_ratio
for col in id_columns:
    print(f'{col}\n', 'n_unique:', data[col].nunique(), '\tna_ratio:', na_ratio[col])
    print("")

In [ ]:
data.drop('pooltypeid7', axis=1, inplace=True)

In [ ]:
id_columns = id_columns.drop('pooltypeid7')

In [ ]:
# # 各カラムの一意な値の種類と個数を表示
# for col in id_columns:
#     val_cnt = data[col].value_counts()
#     print(f"Column: {col}")
#     print(f"n_unique: {data[col].nunique()}")
#     print(val_cnt)
#     print()

変数`id_columns`の対象のカラムの欠損値を頻出値で置き換える。

In [ ]:
data[id_columns].mode().iloc[0]

In [ ]:
_data = data.copy()

In [ ]:
# data = _data.copy()

In [ ]:
data[id_columns] = data[id_columns].fillna(data[id_columns].mode().iloc[0])

ここまでのカラムの欠損値の割合を表示

In [ ]:
na_ratio = data.isna().sum().sort_values(ascending=False)/len(data)
na_ratio

In [ ]:
na_ratio[na_ratio>0]

次に、cntとそうでないカラムの二パターンに分けて考えてみる。cntは個体の数え値であると思われる。

In [ ]:
#欠損値を含む変数を抽出
na_cols = na_ratio[na_ratio>0].index
na_cols

In [ ]:
# ends with 'cnt'
na_cnt = na_cols[na_cols.str.endswith('cnt')]
na_others = na_cols[~na_cols.str.endswith('cnt')]
print("na_cnt \n", na_cnt)
print("na_others \n", na_others)

カラム名にcntを含むカラムを変数`na_cnt`に格納し、一意な値の種類と欠損値の含有率を表示。`fireplacement`と`poolcnt`は欠損値が多く一意な値が一つまたは少ないため特徴量として寄与しにくいので削除する。その他のカラムの欠損値は頻出値で置き換える。

In [ ]:
for col in na_cnt:
    print(f'{col}  d_type:{data[col].dtype}\n', 'n_unique:', data[col].nunique(), '\tna_ratio:', na_ratio[col])
    print("")

In [ ]:
data.drop(['fireplacecnt','poolcnt'], axis=1, inplace=True)
na_cnt = na_cnt.drop(['fireplacecnt','poolcnt'])
data.fillna(data[na_cnt].mode().iloc[0], inplace=True)

その他のカラムも同様に処理する。

In [ ]:
for col in na_others:
    print(f'{col}  d_type:{data[col].dtype}\n', 'n_unique:', data[col].nunique(), '\tna_ratio:', na_ratio[col])
    print("")

In [ ]:
data.drop(['threequarterbathnbr', 'numberofstories'], axis=1, inplace=True)
na_others = na_others.drop(['threequarterbathnbr', 'numberofstories'])
data.fillna(data[na_others].mode().iloc[0], inplace=True)

In [ ]:
data.isna().sum()

na_cnt に入っているカラム

In [ ]:
# fireplacecnt: ファイヤープレイス（暖炉）の数
# poolcnt: プールの数
# garagecarcnt: ガレージの収容可能な車両数
# unitcnt: 物件のユニット数
# fullbathcnt: フルバスルームの数
# structuretaxvaluedollarcnt: 建物の評価額（ドル）
# landtaxvaluedollarcnt: 土地の評価額（ドル）
# taxvaluedollarcnt: 不動産の評価額（ドル）
# bedroomcnt: ベッドルームの数
# bathroomcnt: バスルームの数
# roomcnt: 部屋の総数

na_othersに入っているカラム

In [ ]:
# threequarterbathnbr: 3/4バスルームの数
# numberofstories: 建物の階数
# garagetotalsqft: ガレージの総面積（平方フィート）
# propertyzoningdesc: 物件のゾーニングコード（地域の用途や制限を示すコード）
# calculatedbathnbr: 計算されたバスルームの数
# yearbuilt: 建物の築年数
# calculatedfinishedsquarefeet: 計算された建物の総面積（平方フィート）
# censustractandblock: 国勢調査トラクトとブロックグループの識別子
# taxamount: 不動産税額
# propertycountylandusecode: 物件の郡の土地利用コード
# rawcensustractandblock: 加工されていない国勢調査トラクトとブロックグループの識別子
# assessmentyear: 評価年
# longitude: 物件の経度
# latitude: 物件の緯度
# fips: 物件の連邦情報処理基準コード（地域を識別するための番号）

datatime64型のカラムの処理を行う。以下のように、`year`,`month`,`day`,`weekday`に分解して特徴量として加える。

In [ ]:
data['transactiondate'] = pd.to_datetime(data['transactiondate'], format = '%Y-%m-%d')

data['year'] = data['transactiondate'].dt.year
data['month'] = data['transactiondate'].dt.month
data['day'] = data['transactiondate'].dt.day
data['weekday'] = data['transactiondate'].dt.weekday
data.drop('transactiondate', axis=1, inplace=True)

In [ ]:
data.dtypes.value_counts()

データセット全体を眺めてみる。

In [ ]:
data.info()

最後に、カテゴリカルなカラムをエンコードする。

In [ ]:
data.columns

カラムが表す情報について記載する。

In [ ]:
# 'logerror': この列は、予測価格と実際の価格との差を表す対数誤差を示します。不動産の予測モデルの精度を評価するために使用されます。

# 'airconditioningtypeid': この列は、物件に設置されている空調システムのタイプを示します。 //

# 'bathroomcnt': この列は、物件のバスルームの数を示します。

# 'bedroomcnt': この列は、物件の寝室の数を示します。

# 'buildingqualitytypeid': この列は、建物の品質を示す指標です。 //

# 'calculatedbathnbr': この列は、物件の浴室の数を示します。

# 'calculatedfinishedsquarefeet': この列は、物件の計算された完成した床面積を示します。

# 'fips': この列は、物件の郡の連邦情報処理標準コードを示します。 //

# 'fullbathcnt': この列は、物件のフルバスルームの数を示します。

# 'garagecarcnt': この列は、物件のガレージの車両収容台数を示します。

# 'garagetotalsqft': この列は、物件のガレージの総面積を示します。

# 'hashottuborspa': この列は、物件にホットタブまたはスパがあるかどうかを示します。 //

# 'heatingorsystemtypeid': この列は、物件に設置されている暖房システムのタイプを示します。 //

# 'latitude': この列は、物件の緯度座標を示します。

# 'longitude': この列は、物件の経度座標を示します。

# 'propertycountylandusecode': この列は、物件の郡の土地利用コードを示します。 //

# 'propertylandusetypeid': この列は、物件の用途を示すコードです。 //

# 'propertyzoningdesc': この列は、物件の地域のゾーニングコードや用途に関する説明を示します。 //

# 'rawcensustractandblock': この列は、国勢調査のトラクトとブロックの識別子を示します。 //

# 'regionidcity': この列は、物件が所在する都市の識別子を示します。 //

# 'regionidcounty': この列は、物件が所在する郡の識別子を示します。 //

# 'regionidneighborhood': この列は、物件が所在する近隣地域の識別子を示します。 //

# 'regionidzip': この列は、物件が所在する郵便番号の識別子 //

# 'roomcnt': この列は、物件の部屋の総数を示します。

# 'unitcnt': この列は、物件内のユニット（住宅やアパートなどの独立した住居単位）の数を示します。

# 'yearbuilt': この列は、物件の建築年を示します。

# 'fireplaceflag': この列は、物件に暖炉があるかどうかを示すフラグです。 //

# 'structuretaxvaluedollarcnt': この列は、物件の建物構造の評価額を示します。

# 'taxvaluedollarcnt': この列は、物件の総評価額を示します。

# 'assessmentyear': この列は、物件の評価年を示します。

# 'landtaxvaluedollarcnt': この列は、物件の土地の評価額を示します。

# 'taxamount': この列は、物件の税金額を示します。

# 'taxdelinquencyflag': この列は、物件の税金未納状態を示すフラグです。 //

# 'censustractandblock': この列は、国勢調査のトラクトとブロックの識別子を示します。 //

# 'year': この列は、日付データの年を示します。

# 'month': この列は、日付データの月を示します。

# 'day': この列は、日付データの日を示します。

# 'weekday': この列は、日付データの曜日を示します。

カテゴリカルなカラム(ex: id, タイプ, 個体の種類など)を抽出し、それらをワンホットエンコーディングする。

In [ ]:
# 'airconditioningtypeid': この列は、物件に設置されている空調システムのタイプを示します。 //
# 'buildingqualitytypeid': この列は、建物の品質を示す指標です。 //
# 'fips': この列は、物件の郡の連邦情報処理標準コードを示します。 //
# 'hashottuborspa': この列は、物件にホットタブまたはスパがあるかどうかを示します。 //
# 'heatingorsystemtypeid': この列は、物件に設置されている暖房システムのタイプを示します。 //
# 'propertycountylandusecode': この列は、物件の郡の土地利用コードを示します。 //
# 'propertylandusetypeid': この列は、物件の用途を示すコードです。 //
# 'propertyzoningdesc': この列は、物件の地域のゾーニングコードや用途に関する説明を示します。 //
# 'rawcensustractandblock': この列は、国勢調査のトラクトとブロックの識別子を示します。 //
# 'regionidcity': この列は、物件が所在する都市の識別子を示します。 //
# 'regionidcounty': この列は、物件が所在する郡の識別子を示します。 //
# 'regionidneighborhood': この列は、物件が所在する近隣地域の識別子を示します。 //
# 'regionidzip': この列は、物件が所在する郵便番号の識別子 //
# 'fireplaceflag': この列は、物件に暖炉があるかどうかを示すフラグです。 //
# 'taxdelinquencyflag': この列は、物件の税金未納状態を示すフラグです。 //
# 'censustractandblock': この列は、国勢調査のトラクトとブロックの識別子を示します。 //

In [ ]:
to_encode_col = [
'airconditioningtypeid',
'buildingqualitytypeid',
'fips',
'hashottuborspa',
'heatingorsystemtypeid',
'propertycountylandusecode',
'propertylandusetypeid',
'propertyzoningdesc',
'rawcensustractandblock',
'regionidcity',
'regionidcounty',
'regionidneighborhood',
'regionidzip',
'fireplaceflag',
'taxdelinquencyflag',
'censustractandblock']

続いて、エンコーダであるが、すべてワンホットエンコードすると生成される特徴量が膨大になるのでラベルエンコードを採用する。与えられたカテゴリカルなカラムは数字で与えられているので、文字列やboolで与えられたカラムのみラベルエンコードを適用する。

以下の通り,、`hashottuborspa`, `fireplaceflag`, `taxdelinquencyflag`にラベルエンコードを適用させる。

In [ ]:
for col in to_encode_col:
        unique_values = data[col].unique()
        if data[col].nunique()<10:
            print(f"{col}: {unique_values}")
            print('')

Trueを'True'に置き換える。

In [ ]:
data[['hashottuborspa', 'fireplaceflag']] = data[['hashottuborspa', 'fireplaceflag']].replace({True:'True', True:'True'})

In [ ]:
for col in ['hashottuborspa', 'fireplaceflag', 'taxdelinquencyflag']:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])

残りのカテゴリカルなカラムのエンコードを行う。

In [ ]:
for col in to_encode_col:
    if data[col].dtype == 'object':
        print(f'col_name:{col}  ' ,data[col].nunique())
        print(data[col])

In [ ]:
for col in to_encode_col:
    if data[col].dtype == 'object':
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

カテゴリカルなカラムのエンコードのすべてを完了した。

In [ ]:
for col in to_encode_col:
        unique_values = data[col].unique()
        print(f"{col}: {unique_values}")
        print('')

In [ ]:
import pickle

データ前加工の一連の処理をしたファイルを保存およびダウンロード

In [ ]:
#　w,w+ はファイルをすべて初期化してサイズ0にするので要注意。
# myfile = open("/content/drive/MyDrive/kaggle/zillow_prize/datasets/processing/data_230602.pickle", "w+")

In [ ]:
pd.to_pickle(data, "/content/drive/MyDrive/kaggle/zillow_prize/datasets/processing/data_230602.pickle")

In [ ]:
data = pd.read_pickle("/content/drive/MyDrive/kaggle/zillow_prize/datasets/processing/data_230602.pickle")

In [ ]:
data.info()

ここまでは、欠損値処理と年、月、日の特徴量の追加を行いました。

##EDA

In [ ]:
data.info()

In [ ]:
ax = plt.subplot(1,1,1)
sns.distplot(data['logerror'], color = '#004c70')
plt.title('Overall distribution of Logerror', fontsize = 15)

for s in ['top','left','right']:
    ax.spines[s].set_visible(False)
ax.grid(axis='y', linestyle='-', alpha=0.4)
plt.show()

In [ ]:
plt.subplot(1,2,1)
sns.distplot(data2016['logerror'], color = '#004c70')
plt.title('Year 2016')
plt.subplot(1,2,2)
sns.distplot(data2017['logerror'], color = '#990000')
plt.title('Year 2017')
plt.show()

In [ ]:
color_map = ['#d4dddd' for _ in range(12)]
color_map[10] = color_map[11] = '#004c70'; color_map[4] = color_map[5] = '#990000'


plt.figure(figsize = (12,4))
ax1 = plt.subplot(1,2,1)
data.groupby('month')['logerror'].count().plot.bar(color = color_map)
plt.xticks(rotation = 0); plt.xlabel('Month'); plt.ylabel('Amount')
plt.title('Monthly transaction amount', fontsize = 15)


# axis setting
for s in ["top","right","left"]:
    ax1.spines[s].set_visible(False)
ax1.grid(axis='y', linestyle='-', alpha=0.4)


ax2 = plt.subplot(1,2,2)
data.groupby('month')['logerror'].mean().plot.bar(color = color_map)
plt.axhline(data['logerror'].mean(), linestyle = '--', color = 'black', linewidth = 0.5)
plt.text(0, 0.0145, 'mean', bbox=dict(facecolor='none', edgecolor='black', boxstyle='round'))
plt.xticks(rotation = 0); plt.xlabel('Month'); plt.ylabel('Mean Logerror')
plt.title('Monthly mean of Logerror', fontsize = 15)


# axis setting
for s in ["top","right","left"]:
    ax2.spines[s].set_visible(False)
ax2.grid(axis='y', linestyle='-', alpha=0.4)

plt.tight_layout()

In [ ]:
color_map = ['#d4dddd' for _ in range(7)]
color_map[5] = color_map[6] = '#004c70'

plt.figure(figsize = (12,4))
ax1 = plt.subplot(1,2,1)
data.groupby('weekday')['logerror'].count().plot.bar(color = color_map)
plt.xticks(range(0,7),['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],rotation = 0); plt.xticks(rotation = 0); plt.xlabel('Weekday'); plt.ylabel('Amount')
plt.title('Weekly transaction amount', fontsize = 15)


# axis setting
for s in ["top","right","left"]:
    ax1.spines[s].set_visible(False)
ax1.grid(axis='y', linestyle='-', alpha=0.4)



ax = plt.subplot(1,2,2)
data.groupby('weekday')['logerror'].mean().plot.bar(color = color_map)
plt.axhline(data['logerror'].mean(), linestyle = '--', color = 'black', linewidth = 0.5)
plt.text(6, 0.0145, 'mean', bbox=dict(facecolor='none', edgecolor='black', boxstyle='round'))
plt.xticks(range(0,7),['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],rotation = 0); plt.xlabel('Weekday'); plt.ylabel('Mean Logerror')
plt.title('Weekly mean of Logerror', fontsize = 15)


# axis setting
for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)
ax.grid(axis='y', linestyle='-', alpha=0.4)

plt.tight_layout()

In [ ]:
# yearbuilt.info()

In [ ]:
plt.figure(figsize = (12,4))
ax1 = plt.subplot(1,2,1)
sns.kdeplot(data['yearbuilt'], color = '#004c70');plt.legend(loc='best'); plt.ylabel('Number of residences')
plt.title('Residences built year distribution', fontsize = 15)

# axis setting
for s in ["top","right","left"]:
    ax1.spines[s].set_visible(False)
ax1.grid(axis='y', linestyle='-', alpha=0.4)

ax2 = plt.subplot(1,2,2)
yearbuilt = data.groupby(['yearbuilt'])['logerror'].mean().reset_index()
sns.lineplot(data = yearbuilt, x = 'yearbuilt', y = 'logerror', marker = 'o', markersize = 0.6, color = '#990000')
plt.xticks([1850, 1900, 1950, 2000]); plt.xlabel('Year Built'); plt.ylabel('Mean Logerror')
plt.title('Year Built mean of Logerror', fontsize = 15)
# axis setting
for s in ["top","right","left"]:
    ax2.spines[s].set_visible(False)
ax2.grid(axis='y', linestyle='-', alpha=0.4)

plt.tight_layout()

In [ ]:
data[['latitude','longitude']] = data[['latitude','longitude']]/1000000

In [ ]:
# convert to categorical variable
new_cat = []
for value in data['regionidcounty'].values:
    if value == 1286:
        new_cat.append('Orange')
    elif value == 2061:
        new_cat.append('Ventura')
    else:
        new_cat.append('LA')

data['regionidcounty'] = new_cat

In [ ]:
# county
colors = ['#d4dddd','#004c70', '#990000'] # originally 3, but includes NA values

plt.figure(figsize = (8,4))
ax = plt.subplot(1,1,1)
for i, c in enumerate(data['regionidcounty'].unique()):
    df = data[data['regionidcounty']==c]
    plt.plot(df['longitude'], df['latitude'], 'o', markersize = 0.8, color = colors[i], label = c)
plt.xlabel('Longitude'); plt.ylabel('Latitude'); plt.title('Counties of parcels', fontsize = 15); plt.legend(loc = 'best')

for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)

In [ ]:
corr = pd.DataFrame(data.corr()['logerror'].sort_values(ascending = False)).rename(columns = {'logerror':'correlation'})

plt.figure(figsize = (3,8))
sns.heatmap(corr, annot = True, fmt = '.2f', vmin = -0.05, vmax = 0.05, cmap = 'YlGnBu')
plt.title('Correlation heatmap', fontsize = 15)
plt.show()

## 前処理したデータの読み込み

In [ ]:
data = pd.read_pickle("/content/drive/MyDrive/kaggle/zillow_prize/datasets/processing/data_230602.pickle")

In [ ]:
data.head()

In [ ]:
corr = pd.DataFrame(data.corr()['logerror'].sort_values(ascending = False)).rename(columns = {'logerror':'correlation'})

plt.figure(figsize = (3,8))
sns.heatmap(corr, annot = True, fmt = '.2f', vmin = -0.05, vmax = 0.05, cmap = 'YlGnBu')
plt.title('Correlation heatmap', fontsize = 15)
plt.show()

## モデル
1.pytorchを用いたNNによる線形回帰    
2.xgboostを用いた勾配木ブースティングよる線形回帰  


###1.pytorchを用いたNNによる線形回帰

In [ ]:
!pip install optuna

In [ ]:
import optuna
from optuna.integration import PyTorchIgnitePruningHandler

MLPクラス

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, use_gpu=True):
        super(MLP, self).__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        self.layers_dim = [input_dim] + hidden_dims + [output_dim]
        self.layers = nn.ModuleList()
        self._gpu = use_gpu and torch.cuda.is_available()
        self.eps = 1e-5
        self.num_features = num_features

    def build_model(self):
        for i, dim in enumerate(self.layers_dim):
            if i < len(self.layers_dim)-1:
                new_layer = nn.Linear(dim, self.layers_dim[i+1])
                nn.init.xavier_uniform_(new_layer.weight)
                self.layers.append(new_layer)
                bn_layer = nn.BatchNorm1d(self.layers_dim[i+1], eps=self.eps)
                self.layers.append(bn_layer)

            if i < len(self.layers_dim)-2:
                new_layer = nn.ReLU()
                self.layers.append(new_layer)

    def forward(self, X):
        self.build_model
        if self._gpu:
            self.layers.cuda()

        for layer in self.layers:
            X = layer(X)
        return X


In [ ]:
import os
os.cpu_count()

MyDatasetクラス  
前処理などを加えたカスタムされたデータセットを作成

In [ ]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.X = data.drop('logerror', axis=1)
        self.y = data['logerror']

    def __len__(self):
        return len(self.X), len(self.y)

    def __getitem__(self, idx):
        processed_X = self.X
        processed_y = self.y
        return processed_X, processed_y


 PytorchRegressorクラス

In [ ]:
class PytorchRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=10, hidden_dims =[10, 10], output_dim=1,
                lr_rate=0.01, batch_size=128, use_gpu=True, frac=None, trial_stopper=True,
                num_epochs=10, n_kf=5, n_trials=10, n_startup_trials=5, n_warmup_steps=30, interval_steps=10):
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        self.lr_rate = lr_rate
        self.batch_size = batch_size
        self.use_gpu = use_gpu
        self._gpu = self.use_gpu and torch.cuda.is_available()
        self.frac = frac
        self.trial_stopper = trial_stopper
        self.num_epochs = num_epochs
        self.n_kf = n_kf
        self.n_trials = n_trials
        self.is_pruner = is_pruner
        self.n_startup_trials = n_startup_trials
        self.n_warmup_steps = n_warmup_steps
        self.interval_steps = interval_steps
        self.pruner = None
        if self.is_pruner:
            self.pruner = optuna.pruners.MedianPruner(
                            n_startup_trials=self.n_startup_trials,
                            n_warmup_steps=self.n_warmup_steps, interval_steps=self.interval_steps)


        # # フレームごとに動的に追加したい属性を追加できる
        # args, _, _, values = inspect.getargvalues(inspect.currentframe())
        # values.pop("self")

        # for arg, val in values.items():
        #     setattr(self, arg, val)


    def objective(self, trial):
        self.lr_rate = trial.suggest_float('lr_rate', 1e-5, 1e-1, log=True)
        self.n_hidden = trial.suggest_int('n_hidden', 1, 5, log=False)
        self.hidden_dims = []
        for i in range(self.n_hidden):
            self.hidden_dims.append(trial.suggest_int('hidden_dim_'+str(i+1), 10, 100, log=True))
        self.alpha = trial.suggest_float('alpha', 1e-3, 1e-1, log=True)
        self._model = MLP(input_dim=self.input_dim, hidden_dims=self.hidden_dims, output_dim=self.output_dim)
        self._model.build_model()
        if self._gpu:
            self._model.cuda()

        kf = KFold(n_splits=self.n_kf, shuffle=True, random_state=42)
        loss_fn = nn.MSELoss(reduction='mean')
        optimizer = optim.Adam(self._model.parameters(), lr=self.lr_rate)
        val_loss_kf = []
        cv_cnt = 0

        for train_idx, val_idx in kf.split(self.X):
            if self.trail_stopper and trial.number >= 10:
                raise optuna.exceptions.TrialPruned()
            # if trial.number >= 10:
            #     raise optuna.exceptions.TrialPruned()
            else:
                pass

            if cv_cnt == 0:
                print('--------trail'+str(trial.number+1)+': Start--------')
            print('--------cv'+str(cv_cnt+1)+': Start--------')
            self._model.train()
            X_train, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_train, y_val = self.y.iloc[train_idx], self.y.iloc[val_idx]

            torch_X_train = torch.from_numpy(X_train.values).float()
            torch_y_train = torch.from_numpy(y_train.values).float()
            torch_X_val = torch.from_numpy(X_val.values).float()
            torch_y_val = torch.from_numpy(y_val.values).float()

            torch_X_train.requires_grad_()
            torch_X_val.requires_grad_()
            torch_y_train.requires_grad_()
            torch_y_val.requires_grad_()

            if self._gpu:
                torch_X_train = torch_X_train.cuda()
                torch_X_val = torch_X_val.cuda()
                torch_y_train = torch_y_train.cuda()
                torch_y_val = torch_y_val.cuda()


            train = data_utils.TensorDataset(torch_X_train, torch_y_train)
            train_loader = data_utils.DataLoader(train, batch_size=self.batch_size, shuffle=True)
            for epoch in tqdm(range(self.num_epochs)):
                print('\n--------epoch'+str(epoch+1)+': Start--------')
                current_loss = 0

                for minibatch, target in train_loader:
                    y_pred = self._model(minibatch)
                    loss = loss_fn(y_pred, target.float().unsqueeze(1))
                    l2 = torch.tensor(0., requires_grad=True)
                    for w in self._model.parameters():
                        l2 = l2 + torch.norm(w, p=2)**2
                    loss += self.alpha/2 * l2.item()
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    # 直近の損失
                    current_loss = loss.item()

                # optunaに途中結果を報告
                # minibatchごとに経過をリアルタイムに逐次報告したいならば、一段下げてネスト。
                # これはepochごとで報告
                trial.report(current_loss, step=epoch)
                # 枝狩りのイベント発生したら早期終了
                if trial.should_prune():
                    print("epoch", epoch+1,"で打ち切り")
                    raise optuna.structs.TrialPruned()
                print('\n--------epoch'+str(epoch+1)+': Done--------')

            self._model.eval()
            val_loss = loss_fn(self._model(torch_X_val), torch_y_val.unsqueeze(1))
            val_loss_kf.append(val_loss.item())
            mean_val_loss = np.mean(val_loss_kf)
            print('--------cv'+str(cv_cnt+1)+': Done--------\n')
            cv_cnt += 1

        return mean_val_loss


    def fit(self, X, y, study_name, abs_path='/content/drive/MyDrive/kaggle/zillow_prize/opt_study/torch_save/'):

        self.X = X
        self.y = y
        # 学習データを部分的に使用する場合に使用
        if self.frac:
            self.X = X.sample(frac=self.frac, random_state=42)
            self.y = y.sample(frac=self.frac, random_state=42)

        file_name = study_name + '.db'
        self.study = optuna.create_study(study_name=study_name,
                                    storage='sqlite:///'+abs_path+file_name,
                                    direction="minimize",
                                    load_if_exists=True,
                                    pruner=self.pruner)
                                    # pruner=optuna.pruners.MedianPruner(
                                    # n_startup_trials=self.n_startup_trials, n_warmup_steps=self.n_warmup_steps, interval_steps=self.interval_steps))

        self.study.optimize(self.objective, n_trials=self.n_trials,
                    #    callbacks=[pruning_handler]
                    )
        print('---------ハイパラ探索完了---------')

        best_params = self.study.best_params
        self.lr_rate = best_params['lr_rate']
        self.n_hidden = best_params['n_hidden']
        self.hidden_dims = []
        for i in range(self.n_hidden):
            self.hidden_dims.append(best_params['hidden_dim_'+str(i+1)])
        self._model = MLP(input_dim=self.input_dim, hidden_dims=self.hidden_dims, output_dim=self.output_dim)
        self._model.build_model()
        if self._gpu:
            self._model.cuda()

        loss_fn = nn.MSELoss(reduction='mean')
        self.optimizer = optim.Adam(self._model.parameters(), lr=self.lr_rate)
        self.train_losses = []
        self.val_losses = []
        torch_X_train = torch.from_numpy(X.values).float()
        torch_y_train = torch.from_numpy(y.values).float()
        torch_X_train.requires_grad_()
        torch_y_train.requires_grad_()
        if self._gpu:
            torch_X_train = torch_X_train.cuda()
            torch_y_train = torch_y_train.cuda()

        train = data_utils.TensorDataset(torch_X_train, torch_y_train)
        train_loader = data_utils.DataLoader(train, batch_size=self.batch_size, shuffle=True)
        self._model.train()
        torch.backends.cudnn.benchmark = True
        print('---------学習開始---------')
        for epoch in tqdm(range(self.num_epochs)):
            print('\n--------epoch'+str(epoch+1)+': Start--------')
            train_loss = None
            for minibatch, target in train_loader:
                self.optimizer.zero_grad()
                y_pred = self._model(minibatch)
                train_loss = loss_fn(y_pred, target.float().unsqueeze(1))
                train_loss.backward()
                self.optimizer.step()
            self.train_losses.append(train_loss.item())
            print('\n--------epoch'+str(epoch+1)+' :Done--------')

        print('---------学習完了---------')
        return self


    def predict(self, X):
        torch_X = torch.from_numpy(X.values).float()
        if self._gpu:
            torch_X = torch_X.cuda()
        self._model.eval()
        y_pred = self._model(torch_X).cpu().detach().numpy()
        return y_pred


    def score(self, X, y):
        y = y.to_numpy()
        y_pred = self.predict(X)
        test_loss = mean_absolute_error(y, y_pred)
        return test_loss

    # def show_imortance(self,):
    #     # 特徴量の重要度を参照する
    #     # モデルベースか統計的手法かユースケースかで違ってくる
    #     total_weight = 0.0
    #     feature_importances = []
    #     for param in self._model.parameters():
    #         total_weights += torch.sum(torch.abs(param.data))
    #         feature_importances.append(torch.abs(param.data))

    # def re_fit(self, X, y, ratio=0.6):
    #     # fit()で学習後、特徴量の重要度を算出し、重要度の高い特徴量を指定した割合に絞ってさらに学習を行う

    #     return

    def save(self, file_name, abs_path='/content/drive/MyDrive/kaggle/zillow_prize/save_model/torch_save/'):
        checkpoint = {
            "model_state_dict": self._model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict()
        }
        torch.save(checkpoint, abs_path+file_name+'.pt')

In [ ]:
torch.cuda.device_count()

### 計算の開始

### 一回目
データのレコード数は元の半分で、特徴量は37個
エポック数は10、kfoldは5分割、optunaの試行回数は10回
チューニングするハイパラは、`alpha`: 正則化項パラメータ
`hidden_dim_x`: 隠れ層_xの次元
`lr_rate`: 学習率
`n_hidden`: 隠れ層の数

In [ ]:
y = data['logerror']
X = data.drop('logerror', axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)

Pytorchモデルの学習

In [ ]:
num_features=37
estimator = PytorchRegressor(input_dim=num_features, frac=0.5, num_epochs=10, n_kf=5, n_trials=10)

In [ ]:
study_name = 'opt_study_torch_230605_01'

In [ ]:
estimator.fit(X_train, y_train, study_name)

ハイパラの表示  
`alpha`: 正則化項パラメータ  
`hidden_dim_x`: 隠れ層_xの次元  
`lr_rate`: 学習率  
`n_hidden`: 隠れ層の数

In [ ]:
estimator.study.best_params

スコアの算出

In [ ]:
estimator.score(X_test, y_test)

学習したモデルの保存

In [ ]:
save_model_name = 'torch_model_230605_01'

In [ ]:
estimator.save(save_model_name)

参考として、本コンテストのPublicのトップスコアは0.06318、Privateのトップスコアは0.07408である。

一回目のハイパラ探索から学習までおおよそ一時間程度かかった。  
テスト誤差(スコア)は約0.069であった。Publicスコアとの差はかなり小さく精度のよいモデルを構築できた。  
正則化項パラメータは0.0198、隠れ層は[46]の1層、学習率は0.0057


### 二回目
二回目はデータ数を減らさず元のレコード数で行ってみる。

In [ ]:
num_features=37
estimator = PytorchRegressor(input_dim=num_features, batch_size=512, num_epochs=10, n_kf=5, n_trials=50, n_startup_trials=5, n_warmup_steps=30, interval_steps=5)

In [ ]:
study_name = 'opt_study_torch_230606_01'

In [ ]:
estimator.fit(X_train, y_train, study_name)

ハイパラの表示  
`alpha`: 正則化項パラメータ  
`hidden_dim_x`: 隠れ層_xの次元  
`lr_rate`: 学習率  
`n_hidden`: 隠れ層の数

In [ ]:
estimator.study.best_params

スコアの算出

In [ ]:
estimator.score(X_test, y_test)

学習したモデルの保存

In [ ]:
save_model_name = 'torch_model_230606_01'

In [ ]:
estimator.save(save_model_name)

### 2.xgboostを用いたGBDTよる線形回帰

In [ ]:
class CustomXGBReg(BaseEstimator, RegressorMixin):
    def __init__(self, num_boost_round=10, n_fold=5, n_trials=50,n_startup_trials=10, n_warmup_steps=30, interval_steps=5):
    # def object()のハイパラ
        # general, bosster, learning, command line 各パラメータを設定する
        self.base_params = {
        # ★general parameters
        'booster': 'gbtree',

        #-----------------------------------------------------------------------'
        # ★booster parameters
        'tree_method': 'gpu_hist',

        # 構造上の問題でdef object()内に直接書く
            # # 学習率
            # 'eta': trial.suggest_float('lr_rate', 1e-4, 1e-1, log=True),
            # # 決定木の深さの最大値
            # 'max_depth': trial.suggest_int('max_depth', 3, 8),
            # # 決定木の葉の重みの下限
            # 'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
            # # L1正則化項
            # 'alpha': trial.suggest_float('alpha', 0, 10),
            # # L2正則化項
            # 'lambda': trial.suggest_float('alpha', 0, 10),
        # 決定木の葉の追加におけるペナルティ
        # 分割の前後の評価指標の差がgammaを超える場合にのみ、その分割が行われる。
        # 'gamma':,
        #-----------------------------------------------------------------------'
        # ★learning parameters
        'objective': 'reg:squarederror',
        #緩めの評価ならmae
        'eval_metric': 'rmse'}

        self.num_boost_round = num_boost_round
        self.nfold = n_fold
        self.stratified = False

    #　def fit()のハイパラ
        self.n_trials = n_trials
        self.load_if_exists = True
        self.pruner = optuna.pruners.MedianPruner(
                                        n_startup_trials=n_startup_trials, n_warmup_steps=n_warmup_steps, interval_steps=interval_steps)

    def objective(self, trial):
        d_train = xgb.DMatrix(self.X, label=self.y, nthread=-1)
        trial_params = {
            # 学習率
            'eta': trial.suggest_float('eta', 1e-4, 1e-1, log=True),
            # 決定木の深さの最大値
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            # 決定木の葉の重みの下限
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 100, log=True),
            # L1正則化項
            # 'alpha': trial.suggest_float('alpha', 0, 10),
            # L2正則化項
            'lambda': trial.suggest_float('alpha', 1e-4, 1e-1)
        }
        params = self.base_params | trial_params

        kf = KFold(n_splits=self.nfold, shuffle=True, random_state=0)
        folds = [(train_index, test_index) for train_index, test_index in kf.split(self.X)]
        cv_result = xgb.cv(params=params,
                        dtrain=d_train, num_boost_round=self.num_boost_round,
                        stratified=self.stratified, folds=folds)
                        # metrics=(),obj=None, feval=None, maximize=None,
                        # early_stopping_rounds=5, fpreproc=None,
                        # as_pandas=True, verbose_eval=None,
                        # show_stdv=True, seed=0, callbacks=None,
                        # shuffle=True, custom_metric=None

        score = cv_result['test-{}-mean'.format(self.base_params["eval_metric"])].min()
        return score

    def fit(self, X, y, study_name, abs_path='/content/drive/MyDrive/kaggle/zillow_prize/opt_study/xgb_save/'):
        self.X, self.y = X, y
        self.dtrain = xgb.DMatrix(self.X, self.y)
        file_name = study_name + '.db'
        self.study = optuna.create_study(study_name=study_name,
                                    storage='sqlite:///'+abs_path+file_name,
                                    direction="minimize",
                                    load_if_exists=self.load_if_exists,
                                    pruner=self.pruner)

        self.study.optimize(self.objective, n_trials=self.n_trials)
        best_params = self.study.best_params
        self._model = xgb.train(best_params, dtrain=self.dtrain, num_boost_round=self.num_boost_round)

        # return self

    def predict(self, X):
        dtest = xgb.DMatrix(X)
        y_pred = self._model.predict(dtest)

        return y_pred

    def socre(self, X, y):
        y_pred = self.predict(X)
        test_loss = mean_absolute_error(y, y_pred)

        return test_loss

    def save(self, file_name, abs_path='/content/drive/MyDrive/kaggle/zillow_prize/save_model/xgb_save/'):
        self._model.save_model(abs_path+file_name+'.json')


### 計算の開始

### 一回目
決定木の本数は50本、kfoldの分割数は5、optunaの試行回数は100  
チューニングするハイパラは、`alpha`:正則化項パラメータ,`eta`:学習率,
 `max_depth`:木の深さ,
 `min_child_weight`:決定木の葉の重みの下限

XGBモデルの学習

In [ ]:
estimator_xgb = CustomXGBReg(num_boost_round=50, n_fold=5, n_trials=100)

In [ ]:
study_name = 'opt_study_xgb_230606_01'

In [ ]:
estimator_xgb.fit(X_train, y_train, study_name)

スコアの計算

In [ ]:
estimator_xgb.socre(X_test, y_test)

ハイパラの表示

In [ ]:
estimator_xgb.study.best_params

xgbモデルの保存

In [ ]:
file_name = 'xgb_model_230606_01'

In [ ]:
estimator_xgb.save(file_name)

In [ ]:
cd /content/drive/MyDrive/kaggle/zillow_prize/opt_study/xgb_save/

# まとめ
pytorchによるNNのモデル、xgboostのGBDTモデルはともにスコアは0.069程度になった。NNモデルは、計算リソースの制約上、これ以上深追いはしなかったが、ハイパラ探索のトライアルや学習回数を重ねればさらにスコアをよくできると推測できる。弱点としては、計算時間が多くなることである。効率的にモデルを学習するのに高速化やモデルの複雑性を抑えるなどで工夫する必要性があるだろう。GBDTモデルは、多数の決定木でトライアル回数を多く回しても短時間でNNモデルと同等の精度を実現できた。非常に実用的な手法である。

# 今後の課題
・統計学を中心とした数学とアルゴリズムのより高度な知識を深める   
・論文をサーベイし、実装するスキルを身につける    
・各コンペ(kaggle、atcoderなど)で経験を積む    
・モデル以上にデータの前加工が重要になる場合があるので、実践的で有用なメソッドを学ぶ  
・海外のエンジニアとのレビューでコミュニケーションをとるための英語力の向上  
・自分の得意なドメイン領域を複数開拓する  
・機械学習の実装をサービスに落とし込む  
・環境構築(AWS、Dockerなど)について学び、独自の環境でデプロイし実験できるようにする  
・高速化に関連したフレームワーク(rapids、daskなど)の知識を深める  
・余裕があれば、自前の機械学習用のデスクトップを用意したい  


In [ ]:
特徴量重要度について考察
枝狩りについてxgb